In [1]:
# =========================================
# Cell 1 — Install minimal deps (NO scipy / NO sklearn)
# =========================================
!pip -q install -U "openai>=1.0.0" "datasets>=2.19,<3.0" "huggingface_hub>=0.34,<1.0" pandas tqdm matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
# =========================================
# Cell 2 — Keys (paste in)
# =========================================
import os, getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY (hidden): ").strip()
os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN (required for gated FPB): ").strip()

assert os.environ["OPENAI_API_KEY"], "Missing OPENAI_API_KEY"
assert os.environ["HF_TOKEN"], "Missing HF_TOKEN"
print("✅ Keys set.")


Enter OPENAI_API_KEY (hidden): ··········
Enter HF_TOKEN (required for gated FPB): ··········
✅ Keys set.


In [3]:
# =========================================
# Cell 3 — Load FPB dataset (TheFinAI/en-fpb)
# =========================================
from datasets import load_dataset
import os

DATASET_NAME = "TheFinAI/en-fpb"
SPLIT = "test"   # change if your dataset uses "validation" etc.

ds = load_dataset(DATASET_NAME, split=SPLIT, token=os.environ["HF_TOKEN"])
print(ds)
print("Columns:", ds.column_names)
print("Row0:", ds[0])


Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Row0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


In [4]:
# =========================================
# Cell 4 — o3-mini predictor (OpenAI Responses API) + helpers
# =========================================
import re, time
from openai import OpenAI

MODEL_ID = "o3-mini"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def build_prompt(text: str, choices):
    return (
        "Classify the sentiment of the following financial text.\n"
        "Respond ONLY with one label from the Options.\n\n"
        f"Text: {text}\n"
        f"Options: {', '.join(map(str, choices))}\n"
        "Label:"
    )

def extract_label(model_text: str, choices):
    if not model_text:
        return None
    s = model_text.strip().lower()

    # exact
    for c in choices:
        if s == str(c).strip().lower():
            return str(c)

    # whole word
    for c in choices:
        pat = r"\b" + re.escape(str(c).strip().lower()) + r"\b"
        if re.search(pat, s):
            return str(c)

    return None

def o3mini_predict(prompt: str, max_retries: int = 6, sleep_base: float = 1.0):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.responses.create(model=MODEL_ID, input=prompt)
            text_out = getattr(resp, "output_text", None)

            # make usage JSON-safe
            usage = getattr(resp, "usage", None)
            usage_dict = None
            if usage is not None:
                usage_dict = {
                    "input_tokens": getattr(usage, "input_tokens", None),
                    "output_tokens": getattr(usage, "output_tokens", None),
                    "total_tokens": getattr(usage, "total_tokens", None),
                }

            meta = {"id": getattr(resp, "id", None), "usage": usage_dict}
            return text_out, meta

        except Exception as e:
            last_err = str(e)
            time.sleep(sleep_base * (2 ** attempt))

    return None, {"error": last_err}

def normalize_fpb_example(ex: dict):
    # text field (FPB often uses "sentence" but we check a few)
    text_key = None
    for k in ["text", "sentence", "content", "input"]:
        if k in ex:
            text_key = k
            break
    if text_key is None:
        raise KeyError(f"No text field found. Keys={list(ex.keys())}")
    text = ex[text_key]

    # if already provides choices+gold
    if "choices" in ex and ("gold" in ex or "label" in ex):
        choices = list(ex["choices"])
        gold = int(ex.get("gold", ex.get("label")))
        return text, choices, gold

    # default FPB mapping
    choices = ["negative", "neutral", "positive"]

    lab = ex.get("label", ex.get("gold", ex.get("sentiment", None)))
    if lab is None:
        raise KeyError(f"No label/gold/sentiment found. Keys={list(ex.keys())}")

    if isinstance(lab, str):
        lab_norm = lab.strip().lower()
        if lab_norm not in choices:
            raise ValueError(f"Unknown string label: {lab}")
        gold = choices.index(lab_norm)
    else:
        gold = int(lab)
        # Some FPB variants use 0/1/2; assume it maps to neg/neu/pos
        if gold < 0 or gold >= len(choices):
            raise ValueError(f"Label int {gold} out of range for {choices}")

    return text, choices, gold


In [5]:
# =========================================
# Cell 5 — Run evaluation (full split by default) + resume support
# =========================================
import os, json, random, time
from tqdm import tqdm

SEED = 42
MAX_SAMPLES = None         # None = FULL split
SHUFFLE = True
SLEEP_BETWEEN_CALLS = 0.2  # increase to 0.5–1.0 if rate-limited

random.seed(SEED)

os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"o3-mini_{DATASET_NAME.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_tag}.jsonl"

# resume
done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue

        ex = ds[i]
        text, choices, gold = normalize_fpb_example(ex)

        prompt = build_prompt(text, choices)
        model_out, meta = o3mini_predict(prompt)

        pred_label = extract_label(model_out, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": model_out,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,
        }

        # safe JSON write
        try:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        except TypeError:
            rec["meta"] = {"meta_str": str(meta)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

print("✅ Saved raw responses to:", raw_path)


Already completed: 0


Evaluating: 100%|██████████| 970/970 [52:33<00:00,  3.25s/it]

✅ Saved raw responses to: text_responses/o3-mini_TheFinAI_en-fpb_test.jsonl


In [6]:
# =========================================
# Cell 6 — Evaluation metrics
import json, os

# ---- Load JSONL produced by Cell 5 ----
all_rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try:
                all_rows.append(json.loads(line))
            except:
                pass

assert len(all_rows) > 0, f"No rows found in {raw_path}"

# Infer label names from file (assumes consistent choices)
choices0 = all_rows[0].get("choices", ["negative", "neutral", "positive"])
for r in all_rows[:50]:
    if r.get("choices", choices0) != choices0:
        raise ValueError("Inconsistent 'choices' across rows. Tell me and I’ll adapt this metrics cell.")

idx_to_label = {i: lab for i, lab in enumerate(choices0)}
labels = list(range(len(choices0)))

gold = [int(r["gold"]) for r in all_rows]
pred = [int(r.get("predicted_index", -1)) for r in all_rows]  # -1 means invalid/unparsed

# valid-only for per-class metrics + confusion matrix
pairs = [(g, p) for g, p in zip(gold, pred) if p >= 0]

def safe_div(a, b):
    return a / b if b else 0.0

# ---- Per-class counts (valid-only) ----
support = {c: 0 for c in labels}
tp = {c: 0 for c in labels}
fp = {c: 0 for c in labels}
fn = {c: 0 for c in labels}

for yt, yp in pairs:
    support[yt] += 1
    for c in labels:
        if yp == c and yt == c:
            tp[c] += 1
        elif yp == c and yt != c:
            fp[c] += 1
        elif yp != c and yt == c:
            fn[c] += 1

prec = {}
rec = {}
f1 = {}
for c in labels:
    prec[c] = safe_div(tp[c], tp[c] + fp[c])
    rec[c]  = safe_div(tp[c], tp[c] + fn[c])
    f1[c]   = safe_div(2 * prec[c] * rec[c], prec[c] + rec[c])

# ---- Accuracy ----
acc_all = safe_div(sum(int(a == b) for a, b in zip(gold, pred)), len(gold))                 # invalid counted wrong
acc_valid = safe_div(sum(int(a == b) for a, b in pairs), len(pairs)) if pairs else None    # valid-only

total_support = sum(support.values())

macro_p = sum(prec[c] for c in labels) / len(labels)
macro_r = sum(rec[c]  for c in labels) / len(labels)
macro_f = sum(f1[c]   for c in labels) / len(labels)

w_p = safe_div(sum(prec[c] * support[c] for c in labels), total_support)
w_r = safe_div(sum(rec[c]  * support[c] for c in labels), total_support)
w_f = safe_div(sum(f1[c]   * support[c] for c in labels), total_support)

# ---- Confusion matrix (valid-only) ----
cm = [[0 for _ in labels] for __ in labels]
for g, p in pairs:
    cm[g][p] += 1

# ---- Print like sklearn.classification_report ----
print(f"Accuracy:  {acc_valid:.4f}" if acc_valid is not None else "Accuracy:  N/A (no valid preds)")
print(f"(Accuracy_all_including_invalid_as_wrong: {acc_all:.4f})")
print("\nClassification report:\n")
name_width = max(len(idx_to_label[c]) for c in labels + [labels[0]])
print(f"{'':>{name_width}}  {'precision':>9}  {'recall':>7}  {'f1-score':>8}  {'support':>7}\n")

for c in labels:
    name = idx_to_label[c]
    print(f"{name:>{name_width}}  {prec[c]:9.4f}  {rec[c]:7.4f}  {f1[c]:8.4f}  {support[c]:7d}")

print()
if acc_valid is not None:
    print(f"{'accuracy':>{name_width}}  {'':>9}  {'':>7}  {acc_valid:8.4f}  {total_support:7d}")
else:
    print(f"{'accuracy':>{name_width}}  {'':>9}  {'':>7}  {'N/A':>8}  {total_support:7d}")
print(f"{'macro avg':>{name_width}}  {macro_p:9.4f}  {macro_r:7.4f}  {macro_f:8.4f}  {total_support:7d}")
print(f"{'weighted avg':>{name_width}}  {w_p:9.4f}  {w_r:7.4f}  {w_f:8.4f}  {total_support:7d}")

print("\nConfusion matrix (valid-only) labels:", [idx_to_label[c] for c in labels])
for row in cm:
    print(row)

# ---- Save metrics JSON ----
metrics = {
    "run_tag": run_tag,
    "model_id": MODEL_ID,
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "num_samples": len(all_rows),
    "num_valid_predictions": len(pairs),
    "accuracy_all": acc_all,
    "accuracy_valid_only": acc_valid,
    "per_class": {
        idx_to_label[c]: {
            "precision": prec[c],
            "recall": rec[c],
            "f1": f1[c],
            "support": support[c],
        } for c in labels
    },
    "macro_avg": {"precision": macro_p, "recall": macro_r, "f1": macro_f, "support": total_support},
    "weighted_avg": {"precision": w_p, "recall": w_r, "f1": w_f, "support": total_support},
    "confusion_matrix_labels": [idx_to_label[c] for c in labels],
    "confusion_matrix": cm,
}

os.makedirs("evaluation", exist_ok=True)
metrics_path = f"evaluation/{run_tag}_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\n✅ Metrics saved:", metrics_path)


Accuracy:  0.8113
(Accuracy_all_including_invalid_as_wrong: 0.8113)

Classification report:

          precision   recall  f1-score  support

positive     0.7403   0.6895    0.7140      277
 neutral     0.8353   0.8527    0.8439      577
negative     0.8455   0.8966    0.8703      116

accuracy                        0.8113      970
macro avg     0.8071   0.8129    0.8094      970
weighted avg     0.8094   0.8113    0.8100      970

Confusion matrix (valid-only) labels: ['positive', 'neutral', 'negative']
[191, 85, 1]
[67, 492, 18]
[0, 12, 104]

✅ Metrics saved: evaluation/o3-mini_TheFinAI_en-fpb_test_metrics.json
